In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import xlwings as xw
import re
import time

ModuleNotFoundError: No module named 'selenium'

In [ ]:
# Company name (match Excel exactly) → TradingView exchange-ticker
TICKERS = {
    "Barclays":                                  "LSE-BARC",
    "WBA":                                       "NASDAQ-WBA",
    "Volkswagen":                                "XETR-VOW3",
    "Vodafone (US)":                             "NASDAQ-VOD",
    "Vodafone (British London Stock Exchange)":  "LSE-VOD",
    "Unicaja":                                   "BME-UNI",
    "TRMD-A.CO (Torm PLC)":                      "NASDAQ-TRMD",
    "TLT (iShares 20+ Year Treasuries ETF)":     "NASDAQ-TLT",
    "Telefonica (Spanish stock exchange)":        "BME-TEF",
    "TD":                                        "NYSE-TD",
    "T":                                         "NYSE-T",
    "Societe Generale (GLE.PA)":                 "EURONEXT-GLE",
    "SHEL Plc ADR":                              "NYSE-SHEL",
    "SDR.L (Schroders)":                         "LSE-SDR",
    "SDGR":                                      "NASDAQ-SDGR",
    "SAVA":                                      "NASDAQ-SAVA",
    "SAN.MC":                                    "BME-SAN",
    "RIO":                                       "NYSE-RIO",
    "Repsol":                                    "BME-REP",
    "Renault":                                   "EURONEXT-RNO",
    "Pfizer":                                    "NYSE-PFE",
    "PDM":                                       "NYSE-PDM",
    "PAAS":                                      "NASDAQ-PAAS",
    "P911.DE":                                   "XETR-P911",
    "OXY":                                       "NYSE-OXY",
    "OHLA.MC (OHL Spanish)":                     "BME-OHLA",
    "O (Realty Income Corp)":                    "NYSE-O",
    "NWG.L":                                     "LSE-NWG",
    "MP":                                        "NYSE-MP",
    "Merlin Properties":                         "BME-MRL",
    "Mapfre (MAP.MC)":                           "BME-MAP",
    "LYG (Lloyd's Banking)":                     "NYSE-LYG",
    "Lloyd's (LLOY.L)":                          "LSE-LLOY",
    "KHC":                                       "NASDAQ-KHC",
    "ISP.MI (Intesa Sanpaolo)":                  "MIL-ISP",
    "IRS (Argentinian IRSA)":                    "NYSE-IRS",
    "GRF.MC":                                    "BME-GRF",
    "GLEN.L (Glencore)":                         "LSE-GLEN",
    "GILD":                                      "NASDAQ-GILD",
    "FCC.MC + IMC.MC (Inmocemento)":             "BME-FCC",
    "ENG.MC (Enagás)":                           "BME-ENG",
    "ENB":                                       "NYSE-ENB",
    "DOCS.L (Dr. Hopkins)":                      "LSE-DOCS",
    "DHT":                                       "NYSE-DHT",
    "DB":                                        "NYSE-DB",
    "Danone (BN.PA)":                            "EURONEXT-BN",
    "Citigroup":                                 "NYSE-C",
    "BOXL":                                      "NASDAQ-BOXL",
    "BNS":                                       "NYSE-BNS",
    "BNP Paribas":                               "EURONEXT-BNP",
    "BMO":                                       "NYSE-BMO",
    "BAC":                                       "NYSE-BAC",
    "BABA":                                      "NYSE-BABA",
    "Alibaba (9988.HK)":                         "HKEX-9988",
    "Akzo Nobel (AKZA.NV)":                      "EURONEXT-AKZA",
    "AGG (iShares Core US Aggregate Bond ETF)":  "NYSE-AGG",
    "Acerinox":                                  "BME-ACX",
    "Acciona":                                   "BME-ANA",
    "2899.hk (Zijin Mining Group)":              "HKEX-2899",
    "03993.HK (China Molybdenum Company)":       "HKEX-3993",
    "EVFM":                                      "OTC-EVFM",
}

In [ ]:
def scrape_price(symbol, driver):
    driver.get(f"https://www.tradingview.com/symbols/{symbol}/forecast/")
    time.sleep(6)
    candidates = driver.find_elements(By.CSS_SELECTOR, "[class*='priceWrapper']")
    for el in candidates:
        text = el.text.strip().split('\n')[0]
        if re.match(r'^\d{2,}(\.\d+)?$', text):
            return float(text)
    return None

def update_prices():
    wb = xw.Book.caller()
    ws = wb.sheets["Assets"]

    last_row = ws.range("A1").end("down").row
    ws.range("J1").value = "Price Target"

    driver = webdriver.Safari()
    driver.set_window_size(1280, 900)

    try:
        for row in range(2, last_row + 1):
            company = ws.range(f"A{row}").value
            if not company:
                continue
            symbol = TICKERS.get(str(company).strip())
            if not symbol:
                ws.range(f"J{row}").value = "No ticker"
                continue
            try:
                price = scrape_price(symbol, driver)
                ws.range(f"J{row}").value = price if price else "Not found"
                print(f"{company}: {price}")
            except Exception as e:
                ws.range(f"J{row}").value = "Error"
                print(f"{company} error: {e}")
    finally:
        driver.quit()